# 05 · Other Problem Types

## Colab setup

In [ ]:
import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/neurosat"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

print("Working folder:", os.getcwd())
print("Files:", os.listdir())

In [ ]:
!pip install python-sat -q

In [ ]:
import pickle
import random
import torch
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from pysat.solvers import Minisat22

from neurosat import NeuroSAT, make_batch, to_device

## Settings

In [ ]:
SEED = 42
DIM = 128
BATCH_SIZE = 32
ROUND_LIST = [26, 128]
MIN_NODES = 6
MAX_NODES = 10
N_PER_CLASS = 100

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load model

In [ ]:
model = NeuroSAT(DIM).to(device)
model.load_state_dict(torch.load("neurosat.pt", map_location=device))
model.eval()

## Graphs

In [ ]:
def var(i, v, n):
    return i * n + v + 1


def random_graph(n, p):
    return [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < p]


def is_sat(clauses):
    with Minisat22(bootstrap_with=clauses) as solver:
        return solver.solve()

## Graph colouring

In [ ]:
def coloring(n, edges, k):
    clauses = []
    for v in range(n):
        clauses.append([var(c, v, n) for c in range(k)])
        for c1 in range(k):
            for c2 in range(c1 + 1, k):
                clauses.append([-var(c1, v, n), -var(c2, v, n)])
    for u, v in edges:
        for c in range(k):
            clauses.append([-var(c, u, n), -var(c, v, n)])
    return clauses

## Clique

In [ ]:
def clique(n, edges, k):
    edge_set = set(edges)
    clauses = []
    for i in range(k):
        clauses.append([var(i, v, n) for v in range(n)])
    for v in range(n):
        for i in range(k):
            for j in range(i + 1, k):
                clauses.append([-var(i, v, n), -var(j, v, n)])
    for u in range(n):
        for v in range(u + 1, n):
            if (u, v) not in edge_set:
                for i in range(k):
                    for j in range(k):
                        if i != j:
                            clauses.append([-var(i, u, n), -var(j, v, n)])
    return clauses

## Vertex cover

In [ ]:
def vertex_cover(n, edges, k):
    clauses = []
    for i in range(k):
        clauses.append([var(i, v, n) for v in range(n)])
    for u, v in edges:
        clauses.append([var(i, u, n) for i in range(k)] + [var(i, v, n) for i in range(k)])
    return clauses

## Dominating set

In [ ]:
def dominating_set(n, edges, k):
    neighbours = {v: {v} for v in range(n)}
    for u, v in edges:
        neighbours[u].add(v)
        neighbours[v].add(u)

    clauses = []
    for i in range(k):
        clauses.append([var(i, v, n) for v in range(n)])
    for v in range(n):
        clauses.append([var(i, u, n) for i in range(k) for u in neighbours[v]])
    return clauses

## Datasets

In [ ]:
GENERATORS = {
    "coloring": (coloring, [3, 4]),
    "clique": (clique, [3, 4]),
    "vertex_cover": (vertex_cover, [3, 4, 5, 6]),
    "dominating_set": (dominating_set, [2, 3, 4]),
}


def make_problem(kind):
    build, ks = GENERATORS[kind]
    n = random.randint(MIN_NODES, MAX_NODES)
    edges = random_graph(n, random.uniform(0.2, 0.5))
    k = random.choice(ks)
    clauses = build(n, edges, k)
    return {"n_vars": k * n, "clauses": clauses, "label": int(is_sat(clauses)), "kind": kind}


def make_dataset(kind, n_per_class):
    sat = []
    unsat = []
    while len(sat) < n_per_class or len(unsat) < n_per_class:
        p = make_problem(kind)
        if p["label"] == 1 and len(sat) < n_per_class:
            sat.append(p)
        elif p["label"] == 0 and len(unsat) < n_per_class:
            unsat.append(p)
    return sat + unsat

In [ ]:
datasets = {}
for kind in GENERATORS:
    datasets[kind] = make_dataset(kind, N_PER_CLASS)
    n_vars = [p["n_vars"] for p in datasets[kind]]
    n_clauses = [len(p["clauses"]) for p in datasets[kind]]
    print(f"{kind:15s} | variables {min(n_vars)}–{max(n_vars)} | clauses {min(n_clauses)}–{max(n_clauses)}")

## Accuracy

In [ ]:
def predict(data, rounds):
    preds = []
    with torch.no_grad():
        for i in range(0, len(data), BATCH_SIZE):
            batch = to_device(make_batch(data[i:i + BATCH_SIZE]), device)
            logits = model(batch, rounds)
            preds += (logits > 0).long().tolist()
    return preds


def accuracy(data, rounds):
    preds = predict(data, rounds)
    labels = [p["label"] for p in data]
    return sum(p == l for p, l in zip(preds, labels)) / len(labels)

## Decoding

In [ ]:
def satisfies(clauses, assignment):
    for clause in clauses:
        if not any(assignment[abs(lit) - 1] == (lit > 0) for lit in clause):
            return False
    return True


def decode(problem, rounds):
    batch = to_device(make_batch([problem]), device)
    with torch.no_grad():
        logit, L_h = model(batch, rounds, return_literals=True)

    if logit.item() <= 0:
        return None

    clusters = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(L_h.cpu().numpy())
    n = problem["n_vars"]

    for true_cluster in [0, 1]:
        assignment = [clusters[i] == true_cluster for i in range(n)]
        if satisfies(problem["clauses"], assignment):
            return assignment

    return None


def decode_rate(data, rounds):
    sat_problems = [p for p in data if p["label"] == 1]
    solved = sum(decode(p, rounds) is not None for p in sat_problems)
    return solved / len(sat_problems)

## Run

In [ ]:
results = {}

for kind, data in datasets.items():
    results[kind] = {}
    for rounds in ROUND_LIST:
        results[kind][f"acc_{rounds}"] = accuracy(data, rounds)
        results[kind][f"solved_{rounds}"] = decode_rate(data, rounds)
    print(f"Done {kind}")

## Results table

In [ ]:
header = f"{'problem':>15s}"
for rounds in ROUND_LIST:
    header += f" | {'acc @' + str(rounds):>9s} | {'solved @' + str(rounds):>11s}"
print(header)

for kind, r in results.items():
    row = f"{kind:>15s}"
    for rounds in ROUND_LIST:
        row += f" | {r[f'acc_{rounds}']:9.3f} | {r[f'solved_{rounds}']:11.3f}"
    print(row)

## Plots

In [ ]:
kinds = list(results.keys())
x = range(len(kinds))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for k, rounds in enumerate(ROUND_LIST):
    axes[0].bar([i + k * width for i in x], [results[kind][f"acc_{rounds}"] for kind in kinds], width, label=f"{rounds} rounds")
    axes[1].bar([i + k * width for i in x], [results[kind][f"solved_{rounds}"] for kind in kinds], width, label=f"{rounds} rounds")

axes[0].axhline(0.5, color="gray", linestyle="--")
axes[0].set_ylabel("Accuracy")
axes[1].set_ylabel("Share of SAT problems solved")

for ax in axes:
    ax.set_xticks([i + width / 2 for i in x])
    ax.set_xticklabels(kinds)
    ax.set_ylim(0, 1.05)
    ax.legend()

plt.tight_layout()
plt.savefig("other_problem_types.png", dpi=150)
plt.show()

## Save

In [ ]:
with open("other_problems.pkl", "wb") as f:
    pickle.dump(datasets, f)

with open("other_problem_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved other_problems.pkl, other_problem_results.pkl and other_problem_types.png")